In [1]:
# -*- coding: utf-8 -*-
%load_ext autoreload
%autoreload 2

In [2]:
! pip install json-repair

In [3]:
import json, json_repair
def extract_jsons(response, quite=False):    
    if isinstance(response, (dict, list)):
        # return as it is 
        # if not quite: print("extract_json", "response is already in json format")
        return response       
    elif isinstance(response, str):
        # Method 1
        try:
            # try simple to load it as json
            res = json.loads(response)
            # if not quite: print("extract_json", "response is already in jsons format")
            return res
        except:
            pass
            # if not quite: print("extract_json: simple json load failed. Trying to fix json string ...")
           
        # Method 2 
        try:
            # if not quite: print("extract_json", "response is not in json format. Trying to extract json from response")
            if '```json' in text:                
                out = text.split('```json')[1].split('```')[0].replace('\n','')
            elif '```' in text:
                out = text.split('```')[1].split('```')[0].replace('\n','')
            else:
                out = text

            res = json.loads(out)
            return res        
        except Exception as e:
            # if not quite: print(f"extract_json: unable to fix json string. Trying with json_repair ...")
            pass         
            # it is not in json string format
            
            # Method 3
            text = response
            try:                
                res = json_repair.loads(text)
                if isinstance(res, (dict, list)):
                    # if not quite: print("extract_json: result obtained using repair json")
                    return res
            except:
                if not quite: print("extract_json: unable to repair json string using json_repair. Raise exception")
                raise
    else:
        # if not quite: print("extract_json", "response is not a string or a dictionary")
        return {}  
    

In [57]:
response = """
    Overall, you have demonstrated qualities that align with the role's requirements outlined in the job description. Your performance was Good.

    { "evaluation": { "percentage": "80%", "message": "Good" } }

    I would like to offer you the opportunity for a follow-up interview. If you agree, we can conduct a new round of fresh, diverse questions, including more technical questions related to the specific position you are applying for. Would you like to proceed?
"""
val = extract_json(response)

In [54]:
import re

In [4]:
import json
import json_repair

def extract_json(response, quite=False):    
    if isinstance(response, (dict, list)):
        return response, ""  # Return as it is       
    elif isinstance(response, str):
        # Initialize variables
        json_part = None
        
        # Method 1: Try simple JSON load
        try:
            json_part = json.loads(response)
            return json_part, ""  # Return if already valid JSON with empty text
        except:
            pass
        
        # Method 2: Attempt to extract JSON from response
        try:
            # Find the first occurrence of '{' and the last occurrence of '}'
            start_index = response.index('{')
            end_index = response.rindex('}') + 1
            json_str = response[start_index:end_index]
            json_part = json.loads(json_str)
            # Remove the JSON part from the original response
            non_json_part = response.replace(json_str, '').strip()
            return json_part, non_json_part        
        except Exception:
            pass
            
        # Method 3: Try using json_repair
        try:                
            repaired_json = json_repair.loads(response)
            if isinstance(repaired_json, (dict, list)):
                repaired_json_str = json.dumps(repaired_json)  # Convert to string
                non_json_part = response.replace(repaired_json_str, '').strip()  # Remove JSON part from original response
                return repaired_json, non_json_part
        except:
            if not quite:
                print("extract_json: unable to repair json string using json_repair. Raise exception")
            raise

    # If no valid JSON was found, return None for both
    return None, None

# Example usage
response = """
    Overall, you have demonstrated qualities that align with the role's requirements outlined in the job description. Your performance was Good.

    

    I would like to offer you the opportunity for a follow-up interview. If you agree, we can conduct a new round of fresh, diverse questions, including more technical questions related to the specific position you are applying for. Would you like to proceed?
"""

json_output, other_text = extract_json(response)
print("JSON Output:", json_output)
print("Other Text:", other_text)

JSON Output: None
Other Text: None


In [5]:
# import pandas as pd

# df = pd.read_csv('./dataset/job_match.csv', index_col=0)

# json_data = df.to_json(orient='records', lines=True)

# with open('./dataset/job_match.json', 'w') as json_file:
#     json_file.write(json_data)

# print(json_data)

In [6]:
import pandas as pd
import pprint as pp
df = pd.read_csv('./dataset/extracted/job_match.csv', index_col=0)
len(df)
df.head()

,user_job_match_id,job_profile_id,user_profile_id,match_attributes_overall_match_score,match_attributes_overall_match_degree
0,1558,1035,197,95,very high
1,1557,1041,197,94,very high
2,1556,1082,197,86,high
3,1555,1086,197,96,very high
4,1554,1003,197,97,very high


In [7]:
filtered_df = df[df['match_attributes_overall_match_degree'].isin(['very high'])]
# Assign to result_df
result_df = filtered_df

In [8]:
# Get unique user_profile_ids
unique_user_ids = result_df['user_profile_id'].unique()

# Create a dictionary to store results
user_data = {}

# Fetch 3 rows for each unique user_profile_id
for user_id in unique_user_ids:
    user_data[user_id] = result_df[result_df['user_profile_id'] == user_id].head(3)

# Optionally, convert the dictionary to a DataFrame for better display
final_result_df = pd.concat(user_data.values())

len(final_result_df)

# final_result_df

17

In [9]:
df['match_attributes_overall_match_degree'].unique()

array(['very high', 'high', 'medium', 'very low', 'low'], dtype=object)

In [10]:
result_df['user_profile_id'].unique()

array([197, 167, 176, 183, 189, 180])

In [11]:
result_df['job_profile_id'].unique()

array([1035, 1041, 1086, 1003, 1004, 1045, 1080,  932,  979,  955, 1060,
       1068,  960,  987,  968,  910,  934,  956,  939, 1006, 1040,  914,
        993,  906, 1009,  974,  938,  929,  982,  996, 1011, 1018, 1028,
        989,  983,  905, 1019,  980,  882,  903,  944,  962,  954,  874,
        869,  877,  888,  895,  990,  964,  941,  808,  871,  922,  893,
        827,  918,  781,  935,  815,  897,  793,  901,  972,  971,  836,
        835,  828,  780,  768,  787,  778,  697,  740,  686,  688,  709,
        682,  746,  726,  725,  716,  724,  723,  190,  191,  197,  574,
        616,  618,  621,  414,  584,  386,  470,  415,  416,  369,  479,
        451,  477,  457,  432,  467,   52,   49,  171,  170,  169,  168,
        167,  163,  161,  160,  158,  453,  452,  373,  372,  363,  364,
        318,  320,  301,  280,  260,  341,  359,  365,  296,  274,  298,
        294,  259,  272,  357,  292,  277,  308,  370,  261,  349,  258,
        257,  255,  254,  252,  237,  236,  235])

In [12]:
df2 = pd.read_csv('./dataset/extracted/job_profile.csv', index_col=0)
len(df2)
# df2.head()


1000

In [13]:
job_profile_ids = [1035, 1041, 1086, 808, 929, 616, 190, 191, 49, 197, 618, 621, 52]

filtered_job_profiles = df2[df2['job_profile_id'].isin(job_profile_ids)]

len(filtered_job_profiles)
#filtered_job_profiles['job_profile_id'].unique()

11

In [14]:
df3 = pd.read_csv('./dataset/extracted/user_profile.csv', index_col=0)
len(df3)

30

In [15]:
# Define the user_profile_ids to fetch
user_profile_ids = [197, 167, 176, 183, 189, 180]

filtered_user_profiles = df3[df3['user_profile_id'].isin(user_profile_ids)]

# Display the filtered job profiles
len(filtered_user_profiles)
json_data = filtered_user_profiles.to_json(orient='records', lines=True)

with open('./dataset/final/user_profiles.json', 'w') as json_file:
    json_file.write(json_data)

In [16]:
pp.pprint(len(df))
df.columns.tolist()
'user_job_match_id',
'job_profile_id',
'user_profile_id',
'match_attributes_overall_match_score',
'match_attributes_overall_match_degree',

1000


('match_attributes_overall_match_degree',)

In [17]:
df = pd.read_csv('./dataset/job_match.csv')

# List of columns to keep
columns_to_keep_job_match = [
    'user_job_match_id',
    'job_profile_id',
    'user_profile_id',
    'match_attributes_overall_match_score',
    'match_attributes_overall_match_degree'
]

# Create a new DataFrame with only the specified columns
new_job_match_df = df[columns_to_keep_job_match]
new_job_match_df.to_csv('./dataset/extracted/extracted_job_match.csv')
# Display the new DataFrame
print(new_job_match_df)

     user_job_match_id  job_profile_id  user_profile_id  \
0                 1558            1035              197   
1                 1557            1041              197   
2                 1556            1082              197   
3                 1555            1086              197   
4                 1554            1003              197   
..                 ...             ...              ...   
995                563              75              172   
996                562              74              172   
997                561              73              172   
998                560              86              199   
999                559              78              173   

     match_attributes_overall_match_score  \
0                                      95   
1                                      94   
2                                      86   
3                                      96   
4                                      97   
..                   

In [23]:
df2 = pd.read_csv('./dataset/job_profile.csv')
print(len(df2))
df2.columns.to_list()

1000


['job_profile_id',
 'job_id',
 'title',
 'category',
 'level',
 'label',
 'location',
 'summary',
 'tags',
 'applyLink',
 'job_id.1',
 'role',
 'purpose',
 'apply_link',
 'posted_date',
 'company_info_name',
 'company_info_size',
 'company_info_summary',
 'company_name',
 'competencies_name',
 'competencies_summary',
 'competencies_relevance',
 'competencies_sfia_level',
 'competencies_rationale_name',
 'job_is_a_fit',
 'salary_range',
 'application_url',
 'employment_type',
 'rationale_is_a_fit',
 'job_is_fully_remote',
 'input_is_job_description',
 'job_posted_at_datetime_utc',
 'job_offer_expiration_datetime_utc',
 'createdAt',
 'duties_responsibilities',
 'required_qualifications',
 'preferred_qualifications',
 'url',
 'company_info_location',
 'company_info_mission',
 'company_info_values',
 'company_info_overview',
 'required_qualifications_Education',
 'competencies_rationale_sfia_level',
 'company_info_annual_revenue',
 'competencies_other_attributes',
 'company_info_website',


In [19]:
# Read the CSV file into a DataFrame
df2 = pd.read_csv('./dataset/job_profile.csv')

# List of columns to keep
columns_to_keep_job_profile = [
    'job_profile_id',
    'job_id',
    'title',
    'level',
    'location',
    'summary',
    'applyLink',
    'purpose',
    'posted_date',
    'company_info_name',
    'company_info_size',
    'company_info_summary'
]

# Create a new DataFrame with only the specified columns
new_job_df = df2[columns_to_keep_job_profile]
new_job_df.to_csv('./dataset/extracted/extracted_job_profile.csv')
# Display the new DataFrame
print(new_job_df)

     job_profile_id  job_id                                        title  \
0              1086    8379                     Software Developer Co-op   
1              1085    8378  Compiler Engineer - Distributed ML Training   
2              1084    8377                               Data Scientist   
3              1083    8376                      Python SDET / Developer   
4              1082    8375                                Data Engineer   
..              ...     ...                                          ...   
995              91    7402                                 Data Analyst   
996              90    7379            Data Architect / Sr Data Engineer   
997              89    7392                        Data Analyst (Remote)   
998              88    7380                         Junior Data Engineer   
999              87    7389                           Solutions Engineer   

                    level                    location  \
0             entry level     

In [20]:
df3 = pd.read_csv('./dataset/user_profile.csv')
print(len(df3))
df3.head()

30


,user_profile_id,all_user_id,name,email,batch,trainee_id,category,slug,name.1,awards_code,...,references_contact_full_name,references_contact_relationship,competencies_evidence_remark,competencies_evidence_message_role,competencies_evidence_message_content,competencies_evidence_message_timestamp,competencies_evidence_sfia_dimensions_requested_name,competencies_evidence_sfia_dimensions_requested_level,competencies_evidence_sfia_dimensions_approved_name,competencies_evidence_sfia_dimensions_approved_level
0,199,2104,Eyerusalem Admassu,eyerusad12@gmail.com,7,907,person_profile,"all_user_id:2104, version:1, type:user_profile...",user_profile,awards,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,198,2006,Fanuel Abebe,fanuelabebe@gmail.com,7,652,person_profile,"all_user_id:2006, version:1, type:user_profile...",user_profile,awards,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,197,1959,Abdulhamid Mussa,abdimussa87@gmail.com,7,605,person_profile,"all_user_id:1959, version:1, type:user_profile...",user_profile,awards,...,Belay Birhanu Gib,You Worked together in the same group,NaN,Trainee,test comment,2024-08-25T10:55:07.654Z,Autonomy,4.0,Knowledge,3.0
3,190,1970,Abraham Sahile,abresh.agit@gmail.com,8,898,person_profile,"all_user_id:1970, version:1, type:user_profile...",user_profile,awards,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,189,2249,Yohanes Teshome Kebede,johnteshe13@gmail.com,8,895,person_profile,"all_user_id:2249, version:1, type:user_profile...",user_profile,awards,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
df3.columns.to_list()

['user_profile_id',
 'all_user_id',
 'name',
 'email',
 'batch',
 'trainee_id',
 'category',
 'slug',
 'name.1',
 'awards_code',
 'awards_name',
 'awards_display',
 'awards_template',
 'awards_date',
 'awards_uuid',
 'awards_title',
 'awards_status',
 'awards_awarder',
 'awards_history_blob',
 'awards_history_user',
 'awards_history_status',
 'awards_history_timestamp',
 'awards_summary',
 'awards_verified_by',
 'awards_requested_by',
 'awards_description',
 'awards_profile_type',
 'basics_code',
 'basics_name',
 'basics_display',
 'basics_template',
 'basics_role',
 'basics_uuid',
 'basics_email',
 'basics_image',
 'basics_status',
 'basics_history_blob',
 'basics_history_user',
 'basics_history_status',
 'basics_history_timestamp',
 'basics_location_city',
 'basics_location_icon',
 'basics_location_region',
 'basics_location_address',
 'basics_location_country',
 'basics_location_postal_code',
 'basics_full_name',
 'basics_name_title',
 'basics_verified_by',
 'basics_requested_by',
 

In [38]:
df3.columns.tolist()
"user_profile_id"
"name"
"email"
"basics_location_address"
"basics_personal_statement"
"projects_code"
"projects_name"
"projects_display"
"projects_url"
"projects_history_timestamp"
"projects_summary"
"projects_end_date"
"education_code" 
"education_name"
"education_display"
"education_end_date"
"education_start_date"
"education_study_area"
"education_study_type"
"education_institution_name"
"languages_code"
"languages_display"
"languages_language"
"languages_fluency"
"achievements_code"
"achievements_display"
"certificates_code"
"certificates_name"
"certificates_display"
"certificates_url"
"certificates_issuer"
"work_experience_code"
"work_experience_display"
"work_experience_role"
"work_experience_company"
"work_experience_history_timestamp"
"work_experience_summary"
"work_experience_location"
"work_experience_start_date"
"work_experience_company_url"
       

['user_profile_id',
 'all_user_id',
 'name',
 'email',
 'batch',
 'trainee_id',
 'category',
 'slug',
 'name.1',
 'awards_code',
 'awards_name',
 'awards_display',
 'awards_template',
 'awards_date',
 'awards_uuid',
 'awards_title',
 'awards_status',
 'awards_awarder',
 'awards_history_blob',
 'awards_history_user',
 'awards_history_status',
 'awards_history_timestamp',
 'awards_summary',
 'awards_verified_by',
 'awards_requested_by',
 'awards_description',
 'awards_profile_type',
 'basics_code',
 'basics_name',
 'basics_display',
 'basics_template',
 'basics_role',
 'basics_uuid',
 'basics_email',
 'basics_image',
 'basics_status',
 'basics_history_blob',
 'basics_history_user',
 'basics_history_status',
 'basics_history_timestamp',
 'basics_location_city',
 'basics_location_icon',
 'basics_location_region',
 'basics_location_address',
 'basics_location_country',
 'basics_location_postal_code',
 'basics_full_name',
 'basics_name_title',
 'basics_verified_by',
 'basics_requested_by',
 

In [43]:
import pandas as pd

# Read the CSV file into a DataFrame
df3 = pd.read_csv('./dataset/user_profile.csv')

# List of columns to keep
columns_to_keep = [
    "user_profile_id",
    "name",
    "email",
    "basics_location_address",
    "basics_personal_statement",
    "projects_code",
    "projects_name",
    "projects_display",
    "projects_url",
    "projects_history_timestamp",
    "projects_summary",
    "projects_end_date",
    "education_code",
    "education_name",
    "education_display",
    "education_end_date",
    "education_start_date",
    "education_study_area",
    "education_study_type",
    "education_institution_name",
    "languages_code",
    "languages_display",
    "languages_language",
    "languages_fluency",
    "achievements_code",
    "achievements_display",
    "certificates_code",
    "certificates_name",
    "certificates_display",
    "certificates_url",
    "certificates_issuer",
    "work_experience_code",
    "work_experience_display",
    "work_experience_role",
    "work_experience_company",
    "work_experience_history_timestamp",
    "work_experience_summary",
    "work_experience_location",
    "work_experience_start_date",
    "work_experience_company_url"
]

# Create a new DataFrame with only the specified columns
new_df = df3[columns_to_keep]
new_df.to_csv('./dataset/extracted/extract_user_profile.csv')
# Display the new DataFrame
print(new_df)

    user_profile_id                                 name  \
0               199                   Eyerusalem Admassu   
1               198                         Fanuel Abebe   
2               197                     Abdulhamid Mussa   
3               190                       Abraham Sahile   
4               189               Yohanes Teshome Kebede   
5               188                       Wandera Martin   
6               187                       Tewodros Cheru   
7               186                        Sheila Murugi   
8               185                     Selamawit Tibebu   
9               184                        Nyamusi Moraa   
10              183                       Mistir Nigusse   
11              182                       Melaku Alehegn   
12              181                          Jabez Kassa   
13              180                     Hillary Kipkemoi   
14              179                     Henock Dessalegn   
15              178                     